# Cross-Experiment Comparison — IchiV3 GMX Best Arms

Side-by-side comparison of the top-performing arms from each experiment series (E003–E009).

| Label | Source | Description |
|-------|--------|-------------|
| **E003 vol+liq+whale** | Exp 003 Arm D | Top 75 vol + GMX liquidity filter (500k) + whale caps |
| **E006 15 trades** | Exp 006 Arm B | All pairs + OI cap, 15 max open trades |
| **E007 top50** | Exp 007 Arm C | Top 50 volume, 10 max open trades |
| **E009 top20** | Exp 009 Arm A | Top 20 volume, 10 max open trades |
| **E009 top25** | Exp 009 Arm D | Top 25 volume, 10 max open trades |

All use IchiV3_LS_Static_WhaleCap with OI cap (2.5%) + pool cap (2%) | $100k starting capital | 2021-01-06 to 2026-03-12

In [1]:
import os, json, zipfile, warnings
from pathlib import Path

import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import nest_asyncio

nest_asyncio.apply()
warnings.filterwarnings('ignore')

PROJECT_ROOT = Path('/home/ubuntu/dev/gmx-ccxt-freqtrade')
os.chdir(PROJECT_ROOT)

STARTING_BALANCE = 100_000
TEMPLATE = 'plotly_dark'

ARM_CONFIGS = [
    ('E003 vol+liq+whale', 'experiments/ichiv3-gmx/003-volume-pairlist-layers/results/arm_d_volume_liq_whale.zip', '#FFA15A'),
    ('E006 15 trades',     'experiments/ichiv3-gmx/006-max-open-trades-sweep/results/arm_b_15trades.zip',          '#FECB52'),
    ('E007 top50',         'experiments/ichiv3-gmx/007-pair-universe-sweep/results/arm_c_top50.zip',               '#AB63FA'),
    ('E009 top25',         'experiments/ichiv3-gmx/009-top-volume-focus/results/arm_d_top25.zip',                  '#00CC96'),
    ('E009 top20',         'experiments/ichiv3-gmx/009-top-volume-focus/results/arm_a_top20.zip',                  '#636EFA'),
]

ARM_LABELS = [c[0] for c in ARM_CONFIGS]
ARM_COLORS = [c[2] for c in ARM_CONFIGS]

for label, path, _ in ARM_CONFIGS:
    p = PROJECT_ROOT / path
    print(f'{label}: {p.name} — exists={p.exists()}')

E003 vol+liq+whale: arm_d_volume_liq_whale.zip — exists=True
E006 15 trades: arm_b_15trades.zip — exists=True
E007 top50: arm_c_top50.zip — exists=True
E009 top25: arm_d_top25.zip — exists=True
E009 top20: arm_a_top20.zip — exists=True


In [2]:
# --- Load all result sets ---

def load_trades_from_zip(zip_path):
    with zipfile.ZipFile(zip_path) as z:
        json_name = [n for n in z.namelist() if n.endswith('.json') and 'config' not in n and 'meta' not in n][0]
        with z.open(json_name) as f:
            data = json.load(f)
    strategy_name = list(data['strategy'].keys())[0]
    trades_list = data['strategy'][strategy_name]['trades']
    trades = pd.DataFrame(trades_list)
    trades['open_date'] = pd.to_datetime(trades['open_date'])
    trades['close_date'] = pd.to_datetime(trades['close_date'])
    return trades, strategy_name

all_trades = {}
for label, path, _ in ARM_CONFIGS:
    trades, strategy = load_trades_from_zip(PROJECT_ROOT / path)
    all_trades[label] = trades
    print(f'{label}: {len(trades)} trades loaded ({strategy})')

E003 vol+liq+whale: 982 trades loaded (IchiV3_LS_Static_WhaleCap)
E006 15 trades: 1884 trades loaded (IchiV3_LS_Static_WhaleCap)
E007 top50: 1497 trades loaded (IchiV3_LS_Static_WhaleCap)
E009 top25: 1088 trades loaded (IchiV3_LS_Static_WhaleCap)


E009 top20: 913 trades loaded (IchiV3_LS_Static_WhaleCap)


In [3]:
# --- Side-by-Side Metrics Table ---

def calculate_metrics_raw(trades, starting_balance=STARTING_BALANCE):
    ts = trades.sort_values('close_date').copy()
    ts['cum_profit_abs'] = ts['profit_abs'].cumsum()
    ts['equity'] = starting_balance + ts['cum_profit_abs']
    
    total_profit_abs = ts['profit_abs'].sum()
    total_profit_pct = (total_profit_abs / starting_balance) * 100
    
    days = (ts['close_date'].max() - ts['open_date'].min()).days
    years = days / 365.25
    final_equity = starting_balance + total_profit_abs
    cagr = ((final_equity / starting_balance) ** (1 / years) - 1) * 100 if years > 0 else 0
    
    equity = ts['equity']
    rolling_max = equity.cummax()
    drawdown = (equity - rolling_max) / rolling_max * 100
    max_dd = drawdown.min()
    
    ts['close_day'] = ts['close_date'].dt.date
    daily_pnl = ts.groupby('close_day')['profit_abs'].sum()
    daily_returns = daily_pnl / starting_balance
    
    sharpe = (daily_returns.mean() / daily_returns.std()) * np.sqrt(365) if daily_returns.std() > 0 else 0
    downside = daily_returns[daily_returns < 0]
    sortino = (daily_returns.mean() / downside.std()) * np.sqrt(365) if len(downside) > 0 and downside.std() > 0 else 0
    calmar = abs(cagr / max_dd) if max_dd != 0 else 0
    
    win_rate = (ts['profit_abs'] > 0).mean() * 100
    
    gross_profit = ts.loc[ts['profit_abs'] > 0, 'profit_abs'].sum()
    gross_loss = abs(ts.loc[ts['profit_abs'] < 0, 'profit_abs'].sum())
    profit_factor = gross_profit / gross_loss if gross_loss > 0 else float('inf')
    
    if 'is_short' in ts.columns:
        longs = ts[~ts['is_short']]
        shorts = ts[ts['is_short']]
    else:
        longs = ts[ts['trade_direction'] == 'long']
        shorts = ts[ts['trade_direction'] == 'short']
    
    return {
        'Trades': len(ts),
        'Longs / Shorts': f"{len(longs)} / {len(shorts)}",
        'Unique Pairs': ts['pair'].nunique(),
        'Profit ($)': round(total_profit_abs, 0),
        'Profit (%)': round(total_profit_pct, 1),
        'CAGR (%)': round(cagr, 1),
        'Max DD (%)': round(max_dd, 1),
        'Sharpe': round(sharpe, 2),
        'Sortino': round(sortino, 2),
        'Calmar': round(calmar, 2),
        'Win Rate (%)': round(win_rate, 1),
        'Profit Factor': round(profit_factor, 2),
        'Avg Trade (%)': round(ts['profit_ratio'].mean() * 100, 2),
        'Avg Stake ($)': round(ts['stake_amount'].mean(), 0),
        'Median Stake ($)': round(ts['stake_amount'].median(), 0),
        'Long P&L ($)': round(longs['profit_abs'].sum(), 0),
        'Short P&L ($)': round(shorts['profit_abs'].sum(), 0),
    }

metrics_all = {}
for label in ARM_LABELS:
    metrics_all[label] = calculate_metrics_raw(all_trades[label])

metrics_df = pd.DataFrame(metrics_all)
metrics_df

,E003 vol+liq+whale,E006 15 trades,E007 top50,E009 top25,E009 top20
Trades,982,1884,1497,1088,913
Longs / Shorts,315 / 667,562 / 1322,473 / 1024,365 / 723,315 / 598
Unique Pairs,35,79,63,40,31
Profit ($),212235.0,139265.0,175980.0,119917.0,127754.0
Profit (%),212.2,139.3,176.0,119.9,127.8
CAGR (%),28.1,20.9,24.7,18.7,19.6
Max DD (%),-29.5,-24.3,-26.3,-18.6,-16.4
Sharpe,1.48,1.54,1.8,1.68,2.1
Sortino,3.22,4.03,4.4,3.87,4.71
Calmar,0.95,0.86,0.94,1.01,1.19


In [4]:
# --- Delta Table vs E009 top20 ---

baseline_col = 'E009 top20'
numeric_metrics = metrics_df.loc[metrics_df[baseline_col].apply(lambda x: isinstance(x, (int, float)))]

delta_df = numeric_metrics.copy()
for col in delta_df.columns:
    if col != baseline_col:
        delta_df[col] = delta_df[col] - delta_df[baseline_col]

delta_df[baseline_col] = '(baseline)'

print('Delta vs E009 top20 (positive = higher than baseline):')
delta_df

Delta vs E009 top20 (positive = higher than baseline):


,E003 vol+liq+whale,E006 15 trades,E007 top50,E009 top25,E009 top20
Trades,69,971,584,175,(baseline)
Unique Pairs,4,48,32,9,(baseline)
Profit ($),84481.0,11511.0,48226.0,-7837.0,(baseline)
Profit (%),84.4,11.5,48.2,-7.9,(baseline)
CAGR (%),8.5,1.3,5.1,-0.9,(baseline)
Max DD (%),-13.1,-7.9,-9.9,-2.2,(baseline)
Sharpe,-0.62,-0.56,-0.3,-0.42,(baseline)
Sortino,-1.49,-0.68,-0.31,-0.84,(baseline)
Calmar,-0.24,-0.33,-0.25,-0.18,(baseline)
Win Rate (%),-0.6,-0.7,0.1,-0.4,(baseline)


In [5]:
# --- Overlaid Equity Curves ---

fig = go.Figure()

for label, _, color in ARM_CONFIGS:
    ts = all_trades[label].sort_values('close_date').copy()
    ts['cum_profit_abs'] = ts['profit_abs'].cumsum()
    ts['equity'] = STARTING_BALANCE + ts['cum_profit_abs']
    fig.add_trace(go.Scatter(
        x=ts['close_date'],
        y=ts['equity'],
        mode='lines',
        name=label,
        line=dict(color=color, width=2),
    ))

fig.update_layout(
    title='Equity Curves — Top Experiments Compared',
    xaxis_title='Date',
    yaxis_title='Equity ($)',
    template=TEMPLATE,
    height=600,
    legend=dict(yanchor='top', y=0.99, xanchor='left', x=0.01),
)
fig.show()

In [6]:
# --- Drawdown Comparison ---

fig = go.Figure()

for label, _, color in ARM_CONFIGS:
    ts = all_trades[label].sort_values('close_date').copy()
    ts['cum_profit_abs'] = ts['profit_abs'].cumsum()
    ts['equity'] = STARTING_BALANCE + ts['cum_profit_abs']
    rolling_max = ts['equity'].cummax()
    ts['drawdown_pct'] = (ts['equity'] - rolling_max) / rolling_max * 100
    fig.add_trace(go.Scatter(
        x=ts['close_date'],
        y=ts['drawdown_pct'],
        mode='lines',
        name=label,
        line=dict(color=color, width=1.5),
    ))

fig.update_layout(
    title='Drawdown Comparison — Top Experiments',
    xaxis_title='Date',
    yaxis_title='Drawdown (%)',
    template=TEMPLATE,
    height=500,
    legend=dict(yanchor='bottom', y=0.01, xanchor='left', x=0.01),
)
fig.show()

In [7]:
# --- Long vs Short Profit — Grouped Bar ---

ls_data = []
for label in ARM_LABELS:
    t = all_trades[label].copy()
    if 'is_short' in t.columns:
        t['direction'] = t['is_short'].apply(lambda x: 'Short' if x else 'Long')
    else:
        t['direction'] = t['trade_direction'].apply(lambda x: 'Short' if 'short' in str(x).lower() else 'Long')
    for d in ['Long', 'Short']:
        profit = t.loc[t['direction'] == d, 'profit_abs'].sum()
        ls_data.append({'Experiment': label, 'Direction': d, 'Profit': profit})

ls_df = pd.DataFrame(ls_data)

fig = go.Figure()
for d, color in [('Long', '#636EFA'), ('Short', '#EF553B')]:
    subset = ls_df[ls_df['Direction'] == d]
    fig.add_trace(go.Bar(
        x=subset['Experiment'],
        y=subset['Profit'],
        name=d,
        marker_color=color,
        text=subset['Profit'].apply(lambda x: f'${x:,.0f}'),
        textposition='outside',
    ))

fig.update_layout(
    title='Long vs Short Profit — Cross-Experiment',
    xaxis_title='Experiment',
    yaxis_title='Profit ($)',
    barmode='group',
    template=TEMPLATE,
    height=500,
)
fig.show()

In [8]:
# --- Monthly Returns Heatmap ---

monthly_data = {}
for label in ARM_LABELS:
    ts = all_trades[label].sort_values('close_date').copy()
    ts['month'] = ts['close_date'].dt.to_period('M')
    monthly_pnl = ts.groupby('month')['profit_abs'].sum()
    monthly_ret = (monthly_pnl / STARTING_BALANCE) * 100
    monthly_data[label] = monthly_ret

monthly_df = pd.DataFrame(monthly_data)
monthly_df.index = monthly_df.index.astype(str)
monthly_df = monthly_df.fillna(0)

fig = go.Figure(data=go.Heatmap(
    z=monthly_df.T.values,
    x=monthly_df.index.tolist(),
    y=monthly_df.columns.tolist(),
    colorscale='RdYlGn',
    zmid=0,
    text=np.round(monthly_df.T.values, 1),
    texttemplate='%{text}%',
    textfont=dict(size=7),
))
fig.update_layout(
    title='Monthly Returns (% of Starting Capital) — Cross-Experiment',
    xaxis_title='Month',
    template=TEMPLATE,
    height=400,
    xaxis_tickangle=-45,
)
fig.show()

In [9]:
# --- Parallelism & Capital Efficiency ---

def compute_daily_exposure(trades, starting_balance=STARTING_BALANCE):
    ts = trades.copy()
    ts['open_date'] = pd.to_datetime(ts['open_date'])
    ts['close_date'] = pd.to_datetime(ts['close_date'])

    date_range = pd.date_range(ts['open_date'].min().normalize(),
                               ts['close_date'].max().normalize(), freq='D')

    closed_pnl = ts.groupby(ts['close_date'].dt.normalize())['profit_abs'].sum()
    cum_realized = closed_pnl.cumsum().reindex(date_range, method='ffill').fillna(0)
    balance_series = starting_balance + cum_realized

    records = []
    for day in date_range:
        open_mask = (ts['open_date'].dt.normalize() <= day) & (ts['close_date'].dt.normalize() >= day)
        open_trades = ts[open_mask]
        n_open = len(open_trades)
        total_deployed = open_trades['stake_amount'].sum()
        balance = balance_series.loc[day]
        pct_deployed = (total_deployed / balance * 100) if balance > 0 else 0
        records.append({'date': day, 'open_trades': n_open, 'deployed_pct': pct_deployed})

    return pd.DataFrame(records)

def hex_to_rgba(hex_color, alpha=0.3):
    h = hex_color.lstrip('#')
    r, g, b = int(h[0:2], 16), int(h[2:4], 16), int(h[4:6], 16)
    return f'rgba({r},{g},{b},{alpha})'

daily_exposure = {}
for label, _, color in ARM_CONFIGS:
    daily_exposure[label] = compute_daily_exposure(all_trades[label])
    de = daily_exposure[label]
    print(f'{label}: avg open trades={de["open_trades"].mean():.1f}, '
          f'avg capital deployed={de["deployed_pct"].mean():.0f}%, '
          f'max open trades={de["open_trades"].max()}')

E003 vol+liq+whale: avg open trades=50.4, avg capital deployed=1022%, max open trades=131


E006 15 trades: avg open trades=3.3, avg capital deployed=28%, max open trades=25


E007 top50: avg open trades=16.1, avg capital deployed=269%, max open trades=78


E009 top25: avg open trades=49.0, avg capital deployed=941%, max open trades=137


E009 top20: avg open trades=39.4, avg capital deployed=712%, max open trades=108


In [10]:
# --- Open Trades Over Time ---

fig = make_subplots(rows=len(ARM_LABELS), cols=1, shared_xaxes=True,
                    subplot_titles=ARM_LABELS, vertical_spacing=0.06)

for i, (label, _, color) in enumerate(ARM_CONFIGS, 1):
    de = daily_exposure[label]
    fig.add_trace(go.Scatter(
        x=de['date'], y=de['open_trades'], mode='lines',
        name=label, line=dict(color=color, width=1),
        fill='tozeroy', fillcolor=hex_to_rgba(color, 0.3),
        showlegend=False,
    ), row=i, col=1)
    mean_val = de['open_trades'].mean()
    fig.add_hline(y=mean_val, line_dash='dash', line_color='white', line_width=0.5,
                  annotation_text=f'avg={mean_val:.1f}', annotation_font_color='white',
                  row=i, col=1)
    fig.update_yaxes(title_text='Open', range=[0, 18], row=i, col=1)

fig.update_layout(template=TEMPLATE, height=200 * len(ARM_LABELS),
                  title='Parallelism — Concurrent Open Trades')
fig.show()

In [11]:
# --- Capital Deployed % Over Time ---

fig2 = make_subplots(rows=len(ARM_LABELS), cols=1, shared_xaxes=True,
                     subplot_titles=ARM_LABELS, vertical_spacing=0.06)

for i, (label, _, color) in enumerate(ARM_CONFIGS, 1):
    de = daily_exposure[label]
    deployed_smooth = de['deployed_pct'].rolling(7, min_periods=1).mean()
    fig2.add_trace(go.Scatter(
        x=de['date'], y=deployed_smooth, mode='lines',
        name=label, line=dict(color=color, width=1),
        fill='tozeroy', fillcolor=hex_to_rgba(color, 0.3),
        showlegend=False,
    ), row=i, col=1)
    mean_val = de['deployed_pct'].mean()
    fig2.add_hline(y=mean_val, line_dash='dash', line_color='white', line_width=0.5,
                   annotation_text=f'avg={mean_val:.0f}%', annotation_font_color='white',
                   row=i, col=1)
    fig2.update_yaxes(title_text='% Balance', range=[0, 100], row=i, col=1)

fig2.update_layout(template=TEMPLATE, height=200 * len(ARM_LABELS),
                   title='Capital Efficiency — Stake Deployed as % of Balance (7d smoothed)')
fig2.show()

In [12]:
# --- Pair Overlap Analysis ---

print('=== Unique Pairs Traded per Experiment ===\n')
pair_sets = {}
for label in ARM_LABELS:
    pairs = set(all_trades[label]['pair'].unique())
    pair_sets[label] = pairs
    top_pairs = all_trades[label].groupby('pair')['profit_abs'].sum().sort_values(ascending=False).head(5)
    worst_pairs = all_trades[label].groupby('pair')['profit_abs'].sum().sort_values().head(5)
    print(f'{label}: {len(pairs)} unique pairs')
    print(f'  Top 5 profitable:')
    for pair, profit in top_pairs.items():
        count = len(all_trades[label][all_trades[label]['pair'] == pair])
        print(f'    {pair}: ${profit:,.0f} ({count} trades)')
    print(f'  Bottom 5:')
    for pair, profit in worst_pairs.items():
        count = len(all_trades[label][all_trades[label]['pair'] == pair])
        print(f'    {pair}: ${profit:,.0f} ({count} trades)')
    print()

# Common pairs across all experiments
common = set.intersection(*pair_sets.values())
print(f'\nPairs common to ALL experiments: {len(common)}')
print(sorted(common))

=== Unique Pairs Traded per Experiment ===

E003 vol+liq+whale: 35 unique pairs
  Top 5 profitable:
    SOL/USDC:USDC: $78,088 (85 trades)
    LINK/USDC:USDC: $55,465 (55 trades)
    HYPE/USDC:USDC: $48,161 (19 trades)
    RENDER/USDC:USDC: $31,001 (5 trades)
    PEPE/USDC:USDC: $21,776 (27 trades)
  Bottom 5:
    SUI/USDC:USDC: $-23,636 (5 trades)
    BTC/USDC:USDC: $-20,386 (98 trades)
    AAVE/USDC:USDC: $-12,135 (72 trades)
    ETH/USDC:USDC: $-11,320 (104 trades)
    BNB/USDC:USDC: $-9,743 (31 trades)

E006 15 trades: 79 unique pairs
  Top 5 profitable:
    HYPE/USDC:USDC: $21,634 (19 trades)
    FARTCOIN/USDC:USDC: $15,289 (27 trades)
    NEAR/USDC:USDC: $14,461 (65 trades)
    RENDER/USDC:USDC: $14,460 (17 trades)
    XRP/USDC:USDC: $14,113 (87 trades)
  Bottom 5:
    XMR/USDC:USDC: $-9,856 (21 trades)
    AAVE/USDC:USDC: $-7,128 (111 trades)
    CRV/USDC:USDC: $-6,502 (17 trades)
    ALGO/USDC:USDC: $-6,119 (9 trades)
    BONK/USDC:USDC: $-5,949 (11 trades)

E007 top50: 63 uniq

In [13]:
# --- Rolling Sharpe (90-day) ---

fig = go.Figure()

for label, _, color in ARM_CONFIGS:
    ts = all_trades[label].sort_values('close_date').copy()
    ts['close_day'] = ts['close_date'].dt.date
    daily_pnl = ts.groupby('close_day')['profit_abs'].sum()
    daily_ret = daily_pnl / STARTING_BALANCE
    daily_ret = daily_ret.reindex(pd.date_range(daily_ret.index.min(), daily_ret.index.max(), freq='D'), fill_value=0)
    
    rolling_mean = daily_ret.rolling(90, min_periods=30).mean()
    rolling_std = daily_ret.rolling(90, min_periods=30).std()
    rolling_sharpe = (rolling_mean / rolling_std) * np.sqrt(365)
    
    fig.add_trace(go.Scatter(
        x=rolling_sharpe.index,
        y=rolling_sharpe.values,
        mode='lines',
        name=label,
        line=dict(color=color, width=1.5),
    ))

fig.add_hline(y=0, line_dash='dot', line_color='gray', line_width=0.5)
fig.add_hline(y=2, line_dash='dot', line_color='green', line_width=0.5,
              annotation_text='Sharpe=2', annotation_font_color='green')
fig.update_layout(
    title='Rolling 90-Day Sharpe Ratio — Cross-Experiment',
    xaxis_title='Date',
    yaxis_title='Sharpe Ratio',
    template=TEMPLATE,
    height=500,
    legend=dict(yanchor='top', y=0.99, xanchor='left', x=0.01),
)
fig.show()

In [14]:
# --- Risk-Return Scatter (Sharpe vs Profit, sized by trades) ---

scatter_data = []
for label in ARM_LABELS:
    m = calculate_metrics_raw(all_trades[label])
    scatter_data.append({
        'Experiment': label,
        'Profit ($)': m['Profit ($)'],
        'Sharpe': m['Sharpe'],
        'Max DD (%)': m['Max DD (%)'],
        'Trades': m['Trades'],
        'CAGR (%)': m['CAGR (%)'],
    })

sdf = pd.DataFrame(scatter_data)

fig = go.Figure()
for i, (label, _, color) in enumerate(ARM_CONFIGS):
    row = sdf[sdf['Experiment'] == label].iloc[0]
    fig.add_trace(go.Scatter(
        x=[row['Profit ($)']],
        y=[row['Sharpe']],
        mode='markers+text',
        name=label,
        marker=dict(color=color, size=abs(row['Max DD (%)']) * 2, opacity=0.7,
                    line=dict(width=2, color='white')),
        text=[label],
        textposition='top center',
        textfont=dict(size=10),
    ))

fig.update_layout(
    title='Risk-Return: Sharpe vs Profit (bubble size = drawdown magnitude)',
    xaxis_title='Total Profit ($)',
    yaxis_title='Sharpe Ratio',
    template=TEMPLATE,
    height=500,
    showlegend=False,
)
fig.show()

In [15]:
# --- Yearly Breakdown ---

yearly_data = {}
for label in ARM_LABELS:
    ts = all_trades[label].sort_values('close_date').copy()
    ts['year'] = ts['close_date'].dt.year
    yearly_pnl = ts.groupby('year')['profit_abs'].sum()
    yearly_ret = (yearly_pnl / STARTING_BALANCE) * 100
    yearly_data[label] = yearly_ret

yearly_df = pd.DataFrame(yearly_data).fillna(0)

fig = go.Figure()
for label, _, color in ARM_CONFIGS:
    fig.add_trace(go.Bar(
        x=yearly_df.index.astype(str),
        y=yearly_df[label],
        name=label,
        marker_color=color,
    ))

fig.update_layout(
    title='Yearly Returns (% of Starting Capital)',
    xaxis_title='Year',
    yaxis_title='Return (%)',
    barmode='group',
    template=TEMPLATE,
    height=500,
)
fig.show()

## Summary

**E003 vol+liq+whale wins on raw profit** ($212k vs $128k for E009 top20) because:
- Wider universe (~35 pairs after liquidity filter vs 20) = more trading opportunities
- 982 trades across 35 pairs vs 913 trades across 31 pairs
- But it pays for that with nearly **2x the drawdown** (-29.5% vs -16.4%)

**E009 top20 wins on every risk-adjusted metric:**
- Sharpe: 2.10 vs 1.48 (+42%)
- Sortino: 4.71 vs 3.22 (+46%)
- Calmar: 1.19 vs 0.95 (+25%)
- Max DD: -16.4% vs -29.5% (44% shallower)
- Profit Factor: 1.39 vs 1.27 (+9%)

**The tradeoff is clear:** E003 makes ~$85k more profit but takes on ~2x the risk. Per unit of risk, E009 top20 is significantly more efficient.